In [3]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal, Annotated
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import time
from langgraph.checkpoint.memory import InMemorySaver

In [4]:
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [5]:
def step_1(state: CrashState) -> CrashState:
    print("step 1 executed")
    return {"step1" : "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print(" step 2 hanging.....")
    time.sleep(30)
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print(" step 2 hanging.....")
    return {"step2": "done"}

In [6]:
graph = StateGraph(CrashState)

graph.add_node("step_1", step_1)
graph.add_node("step_2", step_2)
graph.add_node("step_3", step_3)

graph.set_entry_point("step_1")
graph.add_edge("step_1", "step_2")
graph.add_edge("step_2", "step_3")
graph.add_edge("step_3", END)

checkpointer = InMemorySaver()
workflow = graph.compile(checkpointer=checkpointer)


In [7]:
try:
    print("Running graph: please manually interrupt during step2...")
    workflow.invoke({"input": "start"}, config={"configurable": {"thread_id":"thread-1"}})
except KeyboardInterrupt:
    print("Kernel manually interrupted (crash simulated)")  

Running graph: please manually interrupt during step2...
step 1 executed
 step 2 hanging.....
 step 2 hanging.....


In [10]:
print("Rerun- Re-running the graph to show fault tolerance")
final_state = workflow.invoke(None, config={"configurable": {"thread_id":"thread-1"}})
print("Final state: ", final_state )

Rerun- Re-running the graph to show fault tolerance
Final state:  {'input': 'start', 'step1': 'done', 'step2': 'done'}
